# Phase 3, étape 2 : vectoriser et calibrer

Trois exercices, avec la solution juste après chaque énoncé. Essaie d'abord d'écrire ta réponse dans la cellule « À toi », puis compare avec la solution.

**À retenir.** Un tableau NumPy a une `shape` : chaque position est un *axe*. Dans le projet, les axes sont toujours dans cet ordre : `sim` (simulations, en tête), `c` (catégorie), `s` (sexe), `a` (âge). `sum(axis=k)` fait disparaître l'axe `k`. Le *broadcasting* aligne les formes à partir de la droite : un axe de taille 1 (ou absent) est répété.

On travaille sur le cas jouet de la phase 2 (`config/assumptions.yaml`, bloc `toy_example`), ce qui permet de vérifier à la main.

In [1]:
import sys
from pathlib import Path
import numpy as np

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "tests"))

from config import load_config
from helpers import toy_params, toy_pi, toy_state
from demography.engine import step_year

cfg = load_config(ROOT / "config" / "assumptions.yaml")
pi = toy_pi(cfg)
r1 = step_year(toy_state(cfg), toy_params(cfg, 2026), pi, pi, 2026)
N = r1.state.n                       # effectifs au 1er janvier 2027, axes (c, s, a) = (3, 2, 2)
print("N.shape =", N.shape, "-> axes (catégorie, sexe, âge)")
print(N)

N.shape = (3, 2, 2) -> axes (catégorie, sexe, âge)
[[[43.   58.  ]
  [47.   25.65]]

 [[25.8  39.4 ]
  [28.2  36.45]]

 [[17.2  16.6 ]
  [18.8  25.65]]]


## Exercice 1 : une somme pondérée sans boucle

On a un tableau de salaires annuels `w[c, s, a]` (même `shape` que `N`). Calculer la masse salariale $\sum_{c,s,a} N_{c,s,a}\, w_{c,s,a}$ **sans boucle Python**.

In [2]:
# Salaires du cas jouet : privé, public, indépendants ; femmes, hommes ; âges 1 et 2+ (valeurs de test)
w = np.array([[[100.0, 100.0], [120.0, 120.0]],
              [[ 90.0,  90.0], [110.0, 110.0]],
              [[ 72.0,  72.0], [ 88.0,  88.0]]])
assert w.shape == N.shape

# À toi : masse = ...

**Solution.**

In [3]:
# 1) avec trois boucles (référence lente, pour vérifier)
loop = 0.0
for c in range(N.shape[0]):
    for s in range(N.shape[1]):
        for a in range(N.shape[2]):
            loop += N[c, s, a] * w[c, s, a]

# 2) vectorisé : produit terme à terme, puis somme de tous les axes
vect = (N * w).sum()

# 3) équivalent avec einsum : « csa,csa-> » = multiplie les indices identiques, somme tout
ein = np.einsum("csa,csa->", N, w)

print(loop, vect, ein)
assert np.isclose(loop, vect, rtol=0, atol=1e-9) and np.isclose(vect, ein, rtol=0, atol=1e-9)

38142.7 38142.7 38142.700000000004


Variante utile : la masse salariale **par catégorie** garde l'axe `c` et supprime les deux autres.

In [4]:
par_categorie = (N * w).sum(axis=(1, 2))
print(par_categorie, par_categorie.shape)           # (3,)
assert np.isclose(par_categorie.sum(), vect)
# Piège : sum(axis=0) supprime la catégorie, pas l'âge.
print("sum(axis=0).shape =", (N * w).sum(axis=0).shape)

[18818.  12979.5  6345.2] (3,)
sum(axis=0).shape = (2, 2)


## Exercice 2 : 5 simulations de salaire, toujours sans boucle

Les salaires deviennent aléatoires : `w[sim, c, s, a]` avec 5 simulations (ici un facteur multiplicatif tiré par simulation, valeurs de test). Calculer les 5 masses salariales d'un coup, forme attendue `(5,)`.

In [5]:
rng = np.random.default_rng(2026)                    # graine fixe, comme partout dans le projet
facteurs = rng.uniform(0.9, 1.1, size=5)             # (5,)
w_sim = facteurs[:, None, None, None] * w[None, :, :, :]     # (5, 1, 1, 1) x (1, 3, 2, 2) -> (5, 3, 2, 2)
print(w_sim.shape)

# À toi : masses = ...

(5, 3, 2, 2)


**Solution.** `N` a la forme `(3, 2, 2)` : NumPy lui ajoute un axe de tête de taille 1, donc `N * w_sim` a la forme `(5, 3, 2, 2)`.

In [6]:
masses = (N[None, :, :, :] * w_sim).sum(axis=(1, 2, 3))     # on supprime c, s, a ; il reste sim
masses_ein = np.einsum("csa,xcsa->x", N, w_sim)

boucle = np.array([(N * w_sim[i]).sum() for i in range(5)])  # référence lente

print(masses.shape, masses)
assert masses.shape == (5,)
assert np.allclose(masses, masses_ein, rtol=0, atol=1e-9) and np.allclose(masses, boucle, rtol=0, atol=1e-9)
# Le facteur est commun : la masse de chaque simulation vaut facteur x masse de référence (proportionnalité)
assert np.allclose(masses, facteurs * vect, rtol=0, atol=1e-9)

(5,) [35693.44138352 39210.03318118 37893.00568886 37154.80809106
 37035.93108147]


Dans le simulateur, la même idée évite une énorme matrice : l'assiette est `A[sim, t] = G[sim, t] * phi[sim, t] * A_base[t]` (un produit de tableaux `(n_sim, T)` par un vecteur `(T,)`), et les pensions des nouveaux liquidants sont `G @ M` (produit matriciel `(n_sim, T) @ (T, T)`). La boucle ne reste que sur les années.

## Exercice 3 : calibrer par une formule fermée

On cherche $\tau$ tel que $\tau A = B$ ; c'est la calibration de $\tau_0 = B_{2026}/A_{2026}$. Pour 5 simulations d'assiette `A[sim]` et une dépense `B` connue, trouver `tau[sim]`, puis vérifier $D = B - \tau A = 0$.

In [7]:
B = 1200.0
A = masses                                   # on réutilise les 5 masses salariales comme assiettes (valeurs de test)

# À toi : tau = ...

**Solution.** L'équation est linéaire en $\tau$ : pas besoin de solveur.

In [8]:
tau = B / A
D = B - tau * A
print(tau)
assert np.allclose(D, 0.0, atol=1e-9)

# Cas d'erreur : une assiette nulle n'a pas de taux de calibration (le projet lève une erreur, jamais de valeur par défaut)
from actuarial_model.calibration import calibrate_tau0
from config import ConfigError
try:
    calibrate_tau0(B, 0.0)
except ConfigError as e:
    print("erreur attendue :", e)

[0.03361962 0.03060441 0.03166811 0.0322973  0.03240097]
erreur attendue : Assiette nulle en 2026 : tau0 n'est pas défini.


## Pour aller plus loin : la boucle sur les années, pas sur les simulations

La réserve suit $R_{t+1} = R_t(1+r_t) - D_t$ : chaque année dépend de la précédente, donc la boucle sur $t$ est inévitable. En revanche, toutes les simulations avancent ensemble : `R[:, t+1] = R[:, t] * (1 + r[:, t]) - D[:, t]`.

In [9]:
import time
from actuarial_model.finance import reserves

n_sim, T = 10_000, 40
D = np.random.default_rng(1).normal(10.0, 5.0, size=(n_sim, T))
r = np.full((n_sim, T), 0.01)

t0 = time.perf_counter(); R = reserves(0.0, D, r); vec = time.perf_counter() - t0

t0 = time.perf_counter()
R_lent = np.zeros((n_sim, T + 1))
for i in range(n_sim):                       # boucle sur les simulations : à éviter
    for t in range(T):
        R_lent[i, t + 1] = R_lent[i, t] * (1 + r[i, t]) - D[i, t]
lent = time.perf_counter() - t0

assert np.allclose(R, R_lent, rtol=0, atol=1e-9)
print(f"vectorisé : {vec * 1000:.1f} ms ; double boucle : {lent * 1000:.0f} ms ; rapport environ {lent / vec:.0f}")

vectorisé : 13.9 ms ; double boucle : 331 ms ; rapport environ 24


## Ce que je dois savoir refaire seul

- Lire la `shape` d'un tableau et dire quel axe disparaît avec `sum(axis=...)`.
- Prévoir la forme d'un produit avec broadcasting (alignement à droite).
- Remplacer une boucle sur les simulations par un axe de tête.
- Calibrer un paramètre par une formule fermée quand l'équation est linéaire, et vérifier la condition de calibration.